# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

In [1]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [2]:
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


# Before we look at the Artificial Neural Networks

## There is a different kind of Neural Network we could consider

In [4]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [5]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [6]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [7]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


Human predicted 120.0 for an item that actually costs 219.0


In [8]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

# And now - a vanilla Neural Network

During the remainder of this course we will get deeper into how Neural Networks work, and how to train a neural network.

This is just a sneak preview - let's make our own Neural Network, from scratch, using Pytorch.

Use this to get intuition; it's not important to know all about Neural networks at this point..

In [9]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [10]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [11]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [12]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [13]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [14]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 10037.771, Val Loss: 12326.532


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 8305.084, Val Loss: 10713.314


In [15]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [16]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$129 $112 $30 $169 $38 $120 $47 $14 $11 $77 $205 $126 $70 $54 $5 $4 $3 $2 $24 $12 $0 $43 $53 $52 $245 $215 $165 $42 $112 $44 $85 $163 $90 $5 $51 $362 $29 $78 $71 $35 $142 $8 $18 $72 $64 $45 $34 $17 $51 $10 $31 $38 $115 $46 $21 $75 $33 $250 $27 $35 $119 $7 $22 $46 $213 $110 $8 $377 $4 $112 $9 $39 $51 $114 $6 $50 $111 $21 $26 $68 $25 $70 $31 $42 $13 $175 $26 $55 $34 $178 $25 $5 $6 $7 $47 $43 $21 $37 $15 $242 $1 $35 $13 $43 $1 $25 $56 $257 $9 $42 $28 $86 $80 $0 $39 $233 $29 $52 $25 $129 $23 $133 $58 $1 $89 $53 $12 $72 $43 $19 $27 $21 $44 $55 $54 $18 $41 $18 $22 $26 $9 $128 $47 $148 $7 $47 $5 $224 $50 $2 $11 $141 $9 $6 $22 $51 $52 $12 $76 $7 $19 $5 $0 $25 $305 $32 $109 $32 $12 $32 $30 $14 $253 $29 $4 $60 $25 $5 $7 $30 $68 $9 $13 $9 $20 $30 $58 $67 $35 $25 $14 $56 $2 $35 $6 $43 $120 $10 $9 $1 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [17]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [18]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [19]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [20]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [21]:
gpt_4__1_nano(test[0])

'$180'

In [22]:
test[0].price

219.0

In [23]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$39 $34 $30 $10 $20 $80 $24 $65 $11 $870 $363 $121 $5 $19 $29 $8 $71 $5 $40 $31 $54 $26 $65 $45 $182 $304 $355 $5 $501 $64 $50 $15 $10 $55 $35 $81 $60 $26 $34 $13 $175 $45 $25 $105 $40 $5 $57 $3 $65 $52 $20 $105 $225 $0 $147 $16 $8 $80 $48 $3 $116 $28 $46 $40 $179 $39 $90 $295 $25 $74 $17 $8 $30 $4 $0 $21 $126 $5 $9 $3 $30 $3 $10 $74 $11 $0 $68 $56 $30 $4 $13 $25 $5 $20 $2 $78 $9 $7 $30 $325 $50 $3 $7 $11 $51 $82 $10 $370 $19 $99 $0 $636 $54 $43 $4 $180 $0 $7 $64 $47 $19 $311 $50 $16 $0 $10 $10 $51 $29 $64 $49 $13 $65 $5 $85 $5 $55 $0 $53 $62 $6 $0 $0 $12 $134 $48 $5 $90 $15 $18 $1 $144 $22 $4360 $3 $129 $71 $41 $30 $5 $411 $18 $7 $2 $390 $3 $752 $30 $5 $6 $10 $3 $120 $8 $32 $101 $3 $57 $4 $18 $546 $15 $150 $99 $50 $3 $73 $7 $10 $2 $5 $99 $15 $111 $40 $70 $10 $20 $21 $0 

In [24]:
def claude_opus_4_5(item):
    response = completion(model="anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

In [25]:
evaluate(claude_opus_4_5, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$20 $34 $25 $25 $0 $70 $54 $25 $8 $45 $264 $129 $0 $21 $49 $3 $11 $20 $20 $74 $16 $6 $40 $125 $33 $253 $206 $5 $90 $64 $10 $30 $70 $50 $35 $320 $30 $43 $34 $8 $110 $55 $10 $45 $20 $0 $5 $2 $65 $72 $28 $114 $325 $10 $27 $44 $6 $50 $48 $1 $106 $48 $61 $60 $329 $9 $50 $355 $35 $14 $19 $3 $70 $6 $25 $11 $75 $2 $2 $6 $30 $3 $5 $74 $14 $25 $32 $44 $30 $0 $3 $5 $0 $22 $0 $98 $4 $67 $120 $225 $10 $27 $3 $49 $49 $32 $16 $365 $1 $114 $30 $36 $1 $38 $54 $29 $9 $7 $6 $347 $4 $161 $10 $76 $0 $10 $4 $9 $30 $89 $119 $12 $39 $0 $25 $2 $55 $10 $22 $11 $16 $249 $30 $7 $64 $2 $15 $25 $85 $8 $6 $53 $31 $94 $1 $89 $26 $43 $35 $20 $10 $19 $8 $0 $41 $2 $58 $20 $0 $3 $9 $9 $170 $14 $69 $1 $2 $3 $4 $43 $155 $10 $250 $59 $24 $3 $53 $17 $25 $14 $5 $1 $10 $11 $70 $10 $9 $130 $21 $10 

In [26]:
def gemini_3_pro_preview(item):
    response = completion(model="gemini/gemini-3-pro-preview", messages=messages_for(item), reasoning_effort='low')
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_3_pro_preview, test, size=50, workers=2)

In [ ]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_2__5_flash_lite, test)

In [ ]:

def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [ ]:
evaluate(grok_4__1_fast, test)

In [ ]:
# The function for gpt-5.1

def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='high', seed=42)
    return response.choices[0].message.content


In [ ]:
evaluate(gpt_5__1, test)